# DSPP — Dynamic Streaming Pareto Pruning
### Pipeline complet : DSPP-Lite / DSPP-Adaptive, ablations, 4 baselines, 8 streams, toutes les métriques et tous les tests statistiques

Ce notebook reconstruit le pipeline expérimental du papier *"Dynamic Streaming Pareto Pruning"* et l'étend à **3 datasets réels supplémentaires** (CreditCard, HTTP-KDD99, Phishing) en plus des 5 streams d'origine (Electricity, SEA-abrupt, SEA-recurring, Hyperplane-gradual, RBF-incremental).

**Contenu :**
- Méthodes : `DSPP-Lite`, `DSPP-Adaptive`, ablations `StaticSPP` et `StaticSPPFullSearch` (Lite + Adaptive), baselines `OB`, `LB`, `ARF`, `SRP`
- Métriques : accuracy prequential, **Cohen's Kappa**, latence de prédiction (µs), mémoire du pool et du sous-ensemble actif (KB), taille active moyenne, nombre de drifts détectés
- Tests statistiques : **Friedman** (global, streams comme blocs) + **Wilcoxon signé** apparié (Test 1 vs StaticSPP, Test 2 vs StaticSPPFullSearch, Test 3 vs SRP)
- Tables 1 à 4 reproduisant la structure du papier (accuracy, paired tests, gain par stream, efficacité)

**⚠️ Avant de lancer en pleine échelle (10 seeds × 8 streams) :**
- `CreditCard` (~144 Mo) et `HTTP` (~31 Mo) se téléchargent automatiquement au premier accès (via `river`) — prévoir une connexion correcte côté Colab.
- Le run complet peut prendre **plusieurs heures** selon le nombre d'instances retenu par stream (voir `MAX_INSTANCES_CAP` plus bas). Commence par le mode `QUICK_TEST` pour vérifier que tout tourne avant de lancer le run complet.
- Pense à activer un runtime avec plus de RAM si tu montes le nombre de seeds/streams simultanés.


## 1. Installation & imports

In [ ]:
!pip install river scipy pandas numpy matplotlib --quiet

In [ ]:
import itertools
import math
import pickle
import random
import time
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sps

from river import ensemble, forest, tree, drift, metrics as river_metrics
from river import datasets
from river.datasets import synth

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)


## 2. Cœur DSPP

Implémente :
- `WindowBuffer` — erreur pondérée par récence ($\gamma=0.01$) et accord par paire (proxy de diversité inverse), sur le buffer de la fenêtre courante.
- `pareto_front` / `knee_point` — front de Pareto par balayage skyline $O(n \log n)$, sélection du point de coude (max distance perpendiculaire à la corde entre les deux points extrêmes).
- `generate_candidate_subsets` — énumération exhaustive si $|pool| \le 12$, sinon échantillonnage aléatoire de 200 sous-ensembles.
- `eleven_point_scalarization_subset` — variante statique (StaticSPP) : grille de 11 poids de scalarisation $\lambda \in \{0, 0.1, ..., 1.0\}$.
- `DSPPRunner` — moteur unique couvrant `dynamic` (DSPP complet), `static` (StaticSPP) et `static_fullsearch` (StaticSPPFullSearch), pour les variantes `lite` et `adaptive`.


In [ ]:
import itertools
import math
import pickle
import random
import time
from collections import deque

import numpy as np
from river import ensemble, forest, tree, drift, metrics as river_metrics
from river.datasets import synth

In [ ]:
# Config

In [ ]:
WINDOW_SIZE = 200
MAX_POOL = 12
EXHAUSTIVE_THRESHOLD = 12
RANDOM_SUBSET_SAMPLES = 200
GAMMA = 0.01          # recency-weight decay
EVICT_THRESHOLD = 0.5  # weighted-error eviction threshold
N_INJECT = 2           # new base learners injected on drift
GRACE_INSTANCES = 30   # min instances a model needs before scoring counts fully

In [ ]:
# Recency-weighted tracking of error / agreement within a window

In [ ]:
class WindowBuffer:
    """Buffers (x, y, {model_id: pred}) for the current window."""

    def __init__(self):
        self.records = []  # list of (x, y, preds_dict)

    def add(self, x, y, preds):
        self.records.append((x, y, preds))

    def clear(self):
        self.records = []

    def __len__(self):
        return len(self.records)

    def weighted_error(self, model_id):
        n = len(self.records)
        if n == 0:
            return 1.0
        num, den = 0.0, 0.0
        for k, (x, y, preds) in enumerate(self.records):
            w = math.exp(-GAMMA * (n - 1 - k))
            den += w
            p = preds.get(model_id)
            if p is None or p != y:
                num += w
        return num / den if den > 0 else 1.0

    def pairwise_agreement(self, model_id, other_ids):
        if not other_ids:
            return 0.0
        n = len(self.records)
        if n == 0:
            return 0.0
        agree_sum = 0.0
        count = 0
        for oid in other_ids:
            agree = 0
            for (x, y, preds) in self.records:
                pi, pj = preds.get(model_id), preds.get(oid)
                if pi is not None and pj is not None and pi == pj:
                    agree += 1
            agree_sum += agree / n
            count += 1
        return agree_sum / count if count else 0.0

In [ ]:
# Pareto front + knee point

In [ ]:
def pareto_front(candidates):
    """candidates: list of (id, f1, f2). Returns list of ids on the non-dominated
    front (minimizing both f1 and f2), via O(n log n) skyline sweep."""
    if not candidates:
        return []
    ordered = sorted(candidates, key=lambda c: c[1])
    front = []
    running_min_f2 = float("inf")
    for cid, f1, f2 in ordered:
        if f2 < running_min_f2:
            front.append((cid, f1, f2))
            running_min_f2 = f2
    return front


def knee_point(front):
    """Pick the point on the front with max perpendicular distance to the chord
    connecting the two extreme anchor points (min-f1, min-f2)."""
    if len(front) == 1:
        return front[0][0]
    pts = sorted(front, key=lambda c: c[1])
    p1 = pts[0]   # min f1
    p2 = pts[-1]  # max f1 (== min f2 among front, since front sorted by f1)
    x1, y1 = p1[1], p1[2]
    x2, y2 = p2[1], p2[2]
    dx, dy = x2 - x1, y2 - y1
    norm = math.hypot(dx, dy)
    if norm == 0:
        return p1[0]
    best_id, best_dist = p1[0], -1.0
    for cid, f1, f2 in pts:
        dist = abs(dy * f1 - dx * f2 + x2 * y1 - y2 * x1) / norm
        if dist > best_dist:
            best_dist, best_id = dist, cid
    return best_id


def generate_candidate_subsets(model_ids, rng):
    """Enumerate all subsets of size >=2 if |pool| <= threshold, else sample."""
    m = len(model_ids)
    if m < 2:
        return [tuple(model_ids)] if m else []
    if m <= EXHAUSTIVE_THRESHOLD:
        subsets = []
        for r in range(2, m + 1):
            subsets.extend(itertools.combinations(model_ids, r))
        return subsets
    subsets = []
    for _ in range(RANDOM_SUBSET_SAMPLES):
        size = rng.randint(2, m)
        subsets.append(tuple(rng.sample(model_ids, size)))
    return subsets


def eleven_point_scalarization_subset(model_ids, buf):
    """Static baseline (StaticSPP): grid of 11 scalarization weights
    lambda in {0, 0.1, ..., 1.0}, pick best full-pool-derived subset per lambda,
    then knee-select among those 11 (mirrors the original static formulation)."""
    if len(model_ids) < 2:
        return tuple(model_ids)
    subsets = generate_candidate_subsets(model_ids, random.Random(0))
    scored = []
    for s in subsets:
        f1 = float(np.mean([buf.weighted_error(mid) for mid in s]))
        f2 = float(np.mean([buf.pairwise_agreement(mid, [o for o in s if o != mid]) for mid in s]))
        scored.append((s, f1, f2))
    lambdas = [round(i * 0.1, 1) for i in range(11)]
    picks = []
    for lam in lambdas:
        best = min(scored, key=lambda t: lam * t[1] + (1 - lam) * t[2])
        picks.append(best)
    front_candidates = [(s, f1, f2) for (s, f1, f2) in picks]
    front = pareto_front([(s, f1, f2) for (s, f1, f2) in front_candidates])
    if not front:
        front = [(picks[0][0], picks[0][1], picks[0][2])]
    return knee_point(front)


def full_search_subset(model_ids, buf, rng):
    subsets = generate_candidate_subsets(model_ids, rng)
    if not subsets:
        return tuple(model_ids)
    scored = []
    for s in subsets:
        f1 = float(np.mean([buf.weighted_error(mid) for mid in s]))
        f2 = float(np.mean([buf.pairwise_agreement(mid, [o for o in s if o != mid]) for mid in s]))
        scored.append((s, f1, f2))
    front = pareto_front(scored)
    if not front:
        front = [scored[0]]
    return knee_point(front)

In [ ]:
# Base-learner factories

In [ ]:
def make_hoeffding_tree(seed):
    # HoeffdingTreeClassifier has no internal randomness; the per-model seed
    # instead drives that model's private Poisson(1) online-bagging RNG
    # (see DSPPRunner._add_model / _learn_pool for DSPP-Lite diversity injection).
    return tree.HoeffdingTreeClassifier()


def make_hat(seed):
    return tree.HoeffdingAdaptiveTreeClassifier(seed=seed, bootstrap_sampling=True)

In [ ]:
# DSPP engine (covers DSPP-Lite / DSPP-Adaptive / StaticSPP / StaticSPPFullSearch)

In [ ]:
class DSPPRunner:
    """
    variant: 'lite' | 'adaptive'
    mode:    'dynamic' (full DSPP), 'static' (StaticSPP, fixed pool + 11pt wrapper),
             'static_fullsearch' (StaticSPPFullSearch, fixed pool + combinatorial search)
    """

    def __init__(self, variant, mode, seed, init_pool_size=6):
        assert variant in ("lite", "adaptive")
        assert mode in ("dynamic", "static", "static_fullsearch")
        self.variant = variant
        self.mode = mode
        self.seed = seed
        self.rng = random.Random(seed)
        self.np_rng = np.random.RandomState(seed)

        self.pool = {}          # model_id -> classifier
        self.pool_seen = {}     # model_id -> instances seen (grace tracking)
        self.pool_rng = {}      # model_id -> private np.random.RandomState for Poisson(1) bagging (lite only)
        self.next_id = 0
        for _ in range(init_pool_size):
            self._add_model()

        self.active = tuple(self.pool.keys())
        self.buf = WindowBuffer()
        self.adwin = drift.ADWIN()
        self.pending_drift = False

        # metrics
        self.acc_metric = river_metrics.Accuracy()
        self.kappa_metric = river_metrics.CohenKappa()
        self.n_seen = 0
        self.latencies = []  # seconds per prediction
        self.drift_count = 0
        self.active_size_history = []

    def _add_model(self):
        mid = self.next_id
        self.next_id += 1
        seed_i = 100 * self.seed + mid
        if self.variant == "lite":
            self.pool[mid] = make_hoeffding_tree(seed_i)
            self.pool_rng[mid] = np.random.RandomState(seed_i)
        else:
            self.pool[mid] = make_hat(seed_i)
        self.pool_seen[mid] = 0

    def _predict_pool(self, x):
        preds = {}
        for mid, model in self.pool.items():
            try:
                preds[mid] = model.predict_one(x)
            except Exception:
                preds[mid] = None
        return preds

    def _vote(self, preds_active):
        preds_active = {k: v for k, v in preds_active.items() if v is not None}
        if not preds_active:
            return None
        if self.variant == "lite":
            vals, counts = np.unique(list(preds_active.values()), return_counts=True)
            return vals[np.argmax(counts)]
        # adaptive: error-weighted vote
        weights = {}
        for mid in preds_active:
            e = self.buf.weighted_error(mid) if len(self.buf) else 0.5
            weights[mid] = max(1e-3, 1 - e)
        tally = {}
        for mid, p in preds_active.items():
            tally[p] = tally.get(p, 0.0) + weights[mid]
        return max(tally, key=tally.get)

    def _select_active_subset(self):
        model_ids = list(self.pool.keys())
        if len(model_ids) < 2 or len(self.buf) == 0:
            return tuple(model_ids)
        if self.mode == "static":
            return eleven_point_scalarization_subset(model_ids, self.buf)
        # dynamic and static_fullsearch both use combinatorial search + knee
        return full_search_subset(model_ids, self.buf, self.rng)

    def _adapt_pool(self):
        if self.mode != "dynamic":
            return
        if not self.pending_drift:
            return
        self.pending_drift = False
        self.drift_count += 1
        for _ in range(N_INJECT):
            self._add_model()
        # eviction by weighted error threshold
        evictable = [mid for mid in self.pool if self.buf.weighted_error(mid) > EVICT_THRESHOLD]
        for mid in evictable:
            if len(self.pool) > 2:
                del self.pool[mid]
                self.pool_seen.pop(mid, None)
                self.pool_rng.pop(mid, None)
        # hard cap
        if len(self.pool) > MAX_POOL:
            ranked = sorted(self.pool.keys(), key=lambda mid: -self.buf.weighted_error(mid))
            n_excess = len(self.pool) - MAX_POOL
            for mid in ranked[:n_excess]:
                del self.pool[mid]
                self.pool_seen.pop(mid, None)
                self.pool_rng.pop(mid, None)

    def run(self, stream_iter, max_instances=None):
        for i, (x, y) in enumerate(stream_iter):
            if max_instances is not None and i >= max_instances:
                break
            t0 = time.perf_counter()
            preds_all = self._predict_pool(x)
            preds_active = {mid: preds_all.get(mid) for mid in self.active}
            y_pred = self._vote(preds_active)
            t1 = time.perf_counter()
            self.latencies.append(t1 - t0)

            if y_pred is not None:
                self.acc_metric.update(y, y_pred)
                self.kappa_metric.update(y, y_pred)

            self.buf.add(x, y, preds_all)
            for mid, model in self.pool.items():
                if self.variant == "lite":
                    # DSPP-Lite diversity injection: per-model online (Poisson-1)
                    # bagging, each model's own RNG (Oza & Russell, 2001).
                    k = self.pool_rng[mid].poisson(1)
                    for _ in range(k):
                        model.learn_one(x, y)
                else:
                    # DSPP-Adaptive: no external bagging (HAT already
                    # bootstrap-samples internally; stacking degrades accuracy,
                    # see paper Section 6.2).
                    model.learn_one(x, y)
                self.pool_seen[mid] += 1

            correct = 1 if y_pred == y else 0
            self.adwin.update(correct)
            if self.adwin.drift_detected:
                self.pending_drift = True

            self.n_seen += 1
            if self.n_seen % WINDOW_SIZE == 0:
                self._adapt_pool()
                self.active = self._select_active_subset()
                self.active_size_history.append(len(self.active))
                self.buf.clear()

        return self.results()

    def pool_memory_bytes(self):
        total = 0
        for m in self.pool.values():
            try:
                total += len(pickle.dumps(m))
            except Exception:
                pass
        return total

    def active_memory_bytes(self):
        total = 0
        for mid in self.active:
            m = self.pool.get(mid)
            if m is None:
                continue
            try:
                total += len(pickle.dumps(m))
            except Exception:
                pass
        return total

    def results(self):
        lat_us = float(np.mean(self.latencies) * 1e6) if self.latencies else float("nan")
        return {
            "accuracy": self.acc_metric.get(),
            "kappa": self.kappa_metric.get(),
            "latency_us": lat_us,
            "pool_memory_kb": self.pool_memory_bytes() / 1024.0,
            "active_memory_kb": self.active_memory_bytes() / 1024.0,
            "final_pool_size": len(self.pool),
            "mean_active_size": float(np.mean(self.active_size_history)) if self.active_size_history else len(self.active),
            "drift_count": self.drift_count,
        }

In [ ]:
# Baseline wrapper

In [ ]:
def make_baseline(name, seed):
    if name == "OB":
        return ensemble.ADWINBaggingClassifier(model=tree.HoeffdingTreeClassifier(), n_models=10, seed=seed)
    if name == "LB":
        return ensemble.LeveragingBaggingClassifier(model=tree.HoeffdingTreeClassifier(), n_models=10, seed=seed)
    if name == "ARF":
        return forest.ARFClassifier(n_models=10, seed=seed)
    if name == "SRP":
        return ensemble.SRPClassifier(model=tree.HoeffdingTreeClassifier(), n_models=10, seed=seed)
    raise ValueError(name)


def run_baseline(name, stream_iter, seed, max_instances=None):
    model = make_baseline(name, seed)
    acc = river_metrics.Accuracy()
    kappa = river_metrics.CohenKappa()
    latencies = []
    for i, (x, y) in enumerate(stream_iter):
        if max_instances is not None and i >= max_instances:
            break
        t0 = time.perf_counter()
        y_pred = model.predict_one(x)
        t1 = time.perf_counter()
        latencies.append(t1 - t0)
        if y_pred is not None:
            acc.update(y, y_pred)
            kappa.update(y, y_pred)
        model.learn_one(x, y)
    try:
        mem_kb = len(pickle.dumps(model)) / 1024.0
    except Exception:
        mem_kb = float("nan")
    return {
        "accuracy": acc.get(),
        "kappa": kappa.get(),
        "latency_us": float(np.mean(latencies) * 1e6) if latencies else float("nan"),
        "pool_memory_kb": mem_kb,
        "active_memory_kb": mem_kb,
        "final_pool_size": 10,
        "mean_active_size": 10,
        "drift_count": None,
    }

## 3. Baselines streaming (OB, LB, ARF, SRP)

Wrappers autour de `river.ensemble` / `river.forest`, alignés sur les mêmes métriques (accuracy, Kappa, latence, mémoire) que `DSPPRunner`, pour comparaison directe dans les tables.


In [ ]:
# Sanity check — la classe DSPPRunner et run_baseline sont bien définies
assert 'DSPPRunner' in dir()
assert 'run_baseline' in dir()
print('Cœur DSPP + baselines chargés avec succès.')

## 4. Streams — 5 originaux + 3 datasets réels ajoutés

| Stream | Type | Nature du drift | Instances |
|---|---|---|---|
| Electricity | Réel (Elec2) | Récurrent, réel, fréquent | 45 312 |
| SEA-abrupt | Synthétique | Abrupt (`ConceptDriftStream`, width=50) | 5 000 |
| SEA-recurring | Synthétique | Récurrent contrôlé (6 bascules) | 4 800 |
| Hyperplane-gradual | Synthétique | Graduel continu (`mag_change`) | 5 000 |
| RBF-incremental | Synthétique | Incrémental (`RandomRBFDrift`) | 5 000 |
| **CreditCard** *(ajouté)* | **Réel** | Non-stationnarité réelle, fortement déséquilibré (0.17% fraude) | 284 807 |
| **HTTP (KDD99)** *(ajouté)* | **Réel** | Non-stationnarité réelle, fortement déséquilibré (0.4% positif) | 567 498 |
| **Phishing** *(ajouté)* | **Réel** | Stream réel, plus stationnaire (sert de contrôle) | 1 250 |

Ces 3 ajouts répondent directement à la limite soulevée en Discussion (6.1) : l'effet significatif du drift-adaptation n'était démontré que sur **un seul** stream réel (Electricity). CreditCard et HTTP apportent deux streams réels supplémentaires, fortement déséquilibrés, sur lesquels tester si l'effet généralise. Phishing sert de contrôle "peu de drift".


In [ ]:
import itertools
import time

import numpy as np
import pandas as pd
from scipy import stats as sps

from river import datasets
from river.datasets import synth

In [ ]:
# Stream builders

In [ ]:
def sea_recurring(seed, n_switches=6, segment_len=800):
    """Alternates SEA variant 0 / variant 1 every `segment_len` instances,
    `n_switches` times total -- a controlled synthetic analogue of recurring drift."""
    rng_seed = seed
    for i in range(n_switches):
        variant = i % 2
        gen = synth.SEA(variant=variant, seed=rng_seed + i)
        it = iter(gen)
        for _ in range(segment_len):
            yield next(it)


def sea_abrupt(seed):
    base = synth.SEA(variant=0, seed=seed)
    drift = synth.SEA(variant=1, seed=seed)
    # width=50 relative to position=2500 gives a sharp (near-abrupt) transition
    # while avoiding the float-overflow river hits at width=1.
    return iter(synth.ConceptDriftStream(stream=base, drift_stream=drift,
                                          position=2500, width=50, seed=seed))


def hyperplane_gradual(seed):
    return iter(synth.Hyperplane(seed=seed, n_features=10, n_drift_features=2,
                                  mag_change=0.001, noise_percentage=0.05, sigma=0.1))


def rbf_incremental(seed):
    return iter(synth.RandomRBFDrift(seed_model=seed, seed_sample=seed,
                                      n_classes=2, n_features=10, n_centroids=30,
                                      change_speed=0.05, n_drift_centroids=10))


def electricity(seed):
    # real, order-fixed -> seed only affects model-side randomness, not instance order
    return iter(datasets.Elec2())


def creditcard(seed):
    return iter(datasets.CreditCard())


def http_kdd(seed):
    return iter(datasets.HTTP())


def phishing(seed):
    return iter(datasets.Phishing())


STREAM_REGISTRY = {
    # name: (builder(seed) -> iterator, max_instances or None, is_real)
    "Electricity":        (electricity,        45312,  True),
    "SEA-abrupt":         (sea_abrupt,         5000,   False),
    "SEA-recurring":      (sea_recurring,      4800,   False),
    "Hyperplane-gradual": (hyperplane_gradual, 5000,   False),
    "RBF-incremental":    (rbf_incremental,    5000,   False),
    "CreditCard":         (creditcard,         284807, True),
    "HTTP":               (http_kdd,           567498, True),
    "Phishing":           (phishing,           1250,   True),
}

DSPP_CONFIGS = [
    # (label, variant, mode)
    ("DSPP-Lite",              "lite",     "dynamic"),
    ("DSPP-Adaptive",          "adaptive", "dynamic"),
    ("StaticSPP-Lite",         "lite",     "static"),
    ("StaticSPP-Adaptive",     "adaptive", "static"),
    ("StaticSPPFullSearch-Lite",     "lite",     "static_fullsearch"),
    ("StaticSPPFullSearch-Adaptive", "adaptive", "static_fullsearch"),
]
BASELINES = ["OB", "LB", "ARF", "SRP"]

In [ ]:
# Run loop

In [ ]:
def run_all(streams=None, seeds=range(10), max_instances_cap=None, verbose=True):
    """Runs every DSPP config + every baseline on every stream x seed.
    Returns a tidy pandas.DataFrame, one row per (stream, method, seed)."""
    streams = streams or list(STREAM_REGISTRY.keys())
    rows = []
    for stream_name in streams:
        builder, default_cap, is_real = STREAM_REGISTRY[stream_name]
        cap = max_instances_cap if max_instances_cap is not None else default_cap
        for seed in seeds:
            for label, variant, mode in DSPP_CONFIGS:
                t0 = time.time()
                runner = DSPPRunner(variant=variant, mode=mode, seed=seed)
                res = runner.run(builder(seed), max_instances=cap)
                res.update(stream=stream_name, method=label, seed=seed,
                            wall_time_s=time.time() - t0)
                rows.append(res)
                if verbose:
                    print(f"[{stream_name}] {label} seed={seed} acc={res['accuracy']:.4f} "
                          f"({res['wall_time_s']:.1f}s)")
            for name in BASELINES:
                t0 = time.time()
                res = run_baseline(name, builder(seed), seed=seed, max_instances=cap)
                res.update(stream=stream_name, method=name, seed=seed,
                            wall_time_s=time.time() - t0)
                rows.append(res)
                if verbose:
                    print(f"[{stream_name}] {name} seed={seed} acc={res['accuracy']:.4f} "
                          f"({res['wall_time_s']:.1f}s)")
    return pd.DataFrame(rows)

In [ ]:
# Table 1: mean prequential accuracy by stream x method

In [ ]:
def table1_accuracy(df):
    return df.pivot_table(index="stream", columns="method", values="accuracy", aggfunc="mean")


def table1_kappa(df):
    return df.pivot_table(index="stream", columns="method", values="kappa", aggfunc="mean")

In [ ]:
# Friedman test across methods (streams as blocks, mean accuracy per stream)

In [ ]:
def friedman_test(df, methods):
    pivot = df[df.method.isin(methods)].pivot_table(
        index="stream", columns="method", values="accuracy", aggfunc="mean")
    pivot = pivot.dropna()
    stat, p = sps.friedmanchisquare(*[pivot[m].values for m in methods])
    return {"chi2": stat, "p": p, "n_blocks": len(pivot), "methods": methods}

In [ ]:
# Table 2: paired Wilcoxon tests (Test1 / Test2 / Test3), dataset x seed pairs

In [ ]:
def _paired_values(df, method_a, method_b):
    a = df[df.method == method_a].set_index(["stream", "seed"])["accuracy"]
    b = df[df.method == method_b].set_index(["stream", "seed"])["accuracy"]
    common = a.index.intersection(b.index)
    return a.loc[common].values, b.loc[common].values, common


def wilcoxon_paired(df, method_a, method_b, alternative="greater"):
    a, b, idx = _paired_values(df, method_a, method_b)
    if len(a) == 0:
        return None
    diff = a - b
    wins = int((diff > 0).sum())
    n = len(diff)
    try:
        w_stat, p = sps.wilcoxon(a, b, alternative=alternative, zero_method="wilcox")
    except ValueError:
        w_stat, p = float("nan"), float("nan")
    return {
        "comparison": f"{method_a} vs {method_b}",
        "n_pairs": n,
        "wins": wins,
        "mean_gain": float(diff.mean()),
        "W": w_stat,
        "p": p,
    }


def table2_paired_tests(df):
    results = []
    for variant, dspp, static, staticfs in [
        ("Lite", "DSPP-Lite", "StaticSPP-Lite", "StaticSPPFullSearch-Lite"),
        ("Adaptive", "DSPP-Adaptive", "StaticSPP-Adaptive", "StaticSPPFullSearch-Adaptive"),
    ]:
        r1 = wilcoxon_paired(df, dspp, static)
        r1["test"] = "Test1 (vs StaticSPP)"
        r1["variant"] = variant
        r2 = wilcoxon_paired(df, dspp, staticfs)
        r2["test"] = "Test2 (vs StaticSPPFullSearch)"
        r2["variant"] = variant
        r3 = wilcoxon_paired(df, dspp, "SRP")
        r3["test"] = "Test3 (vs SRP)"
        r3["variant"] = variant
        results.extend([r1, r2, r3])
    out = pd.DataFrame(results)
    return out[["test", "variant", "comparison", "n_pairs", "wins", "mean_gain", "W", "p"]]

In [ ]:
# Table 3: per-stream Test2 gain decomposition

In [ ]:
def table3_per_stream_gain(df, variant="Lite"):
    dspp = f"DSPP-{variant}"
    staticfs = f"StaticSPPFullSearch-{variant}"
    a = df[df.method == dspp].groupby("stream")["accuracy"].mean()
    b = df[df.method == staticfs].groupby("stream")["accuracy"].mean()
    return (a - b).rename(f"gain_{variant}").to_frame()

In [ ]:
# Table 4: efficiency (macro-averaged over streams, mean over seeds)

In [ ]:
def table4_efficiency(df):
    return df.groupby("method").agg(
        accuracy=("accuracy", "mean"),
        latency_us=("latency_us", "mean"),
        pool_memory_kb=("pool_memory_kb", "mean"),
        active_memory_kb=("active_memory_kb", "mean"),
        mean_active_size=("mean_active_size", "mean"),
    ).round(3)

## 5. Configuration du run

`QUICK_TEST=True` : run rapide (2 seeds, streams synthétiques courts + Phishing, peu d'instances) pour vérifier que tout s'exécute sans erreur — quelques minutes.

`QUICK_TEST=False` : run complet à l'échelle du papier — **10 seeds × 8 streams**, tous les baselines et toutes les variantes DSPP. CreditCard/HTTP se téléchargent au premier accès. Prévoir plusieurs heures.


In [ ]:
QUICK_TEST = True  # passer à False pour le run complet à l'échelle du papier

if QUICK_TEST:
    STREAMS_TO_RUN = ['SEA-abrupt', 'Hyperplane-gradual', 'Phishing']
    SEEDS = range(2)
    MAX_INSTANCES_CAP = 800   # instances par stream, pour un test rapide
else:
    STREAMS_TO_RUN = list(STREAM_REGISTRY.keys())  # les 8 streams
    SEEDS = range(10)
    MAX_INSTANCES_CAP = None  # utilise le nombre d'instances complet par stream (voir STREAM_REGISTRY)

print('Streams:', STREAMS_TO_RUN)
print('Seeds:', list(SEEDS))
print('Cap instances:', MAX_INSTANCES_CAP)


## 6. Lancer l'expérience

In [ ]:
t_start = time.time()
results_df = run_all(streams=STREAMS_TO_RUN, seeds=SEEDS,
                      max_instances_cap=MAX_INSTANCES_CAP, verbose=True)
print(f"\nTerminé en {(time.time()-t_start)/60:.1f} min — {len(results_df)} lignes.")
results_df.head()


In [ ]:
# Sauvegarde brute (une ligne par stream x méthode x seed) — à garder pour
# ré-analyser sans tout relancer.
results_df.to_csv('dspp_raw_results.csv', index=False)
print('Sauvegardé : dspp_raw_results.csv')


## 7. Table 1 — Accuracy moyenne (prequential) par stream × méthode

In [ ]:
t1_acc = table1_accuracy(results_df)
t1_acc.round(3)

### Table 1bis — Cohen's Kappa moyen par stream × méthode
(Kappa complète l'accuracy brute, en particulier utile sur CreditCard/HTTP qui sont fortement déséquilibrés — l'accuracy seule y est peu informative.)

In [ ]:
t1_kappa = table1_kappa(results_df)
t1_kappa.round(3)

## 8. Test de Friedman global (streams comme blocs)

In [ ]:
dspp_and_baselines = ['DSPP-Lite', 'DSPP-Adaptive', 'StaticSPP-Lite', 'OB', 'LB', 'ARF', 'SRP']
friedman_lite = friedman_test(results_df, dspp_and_baselines)
print('Friedman (méthodes principales, streams=blocs):')
print(friedman_lite)


## 9. Table 2 — Tests appariés de Wilcoxon (Test 1 / Test 2 / Test 3)

- **Test 1** : DSPP vs StaticSPP (effet combiné : recherche combinatoire + adaptation drift)
- **Test 2** : DSPP vs StaticSPPFullSearch (isole l'effet marginal de l'adaptation drift, recherche tenue constante — comparaison critique)
- **Test 3** : DSPP vs SRP (meilleur baseline en accuracy brute)


In [ ]:
t2 = table2_paired_tests(results_df)
t2


## 10. Table 3 — Décomposition du gain (Test 2) par stream

Vérifie si l'effet de l'adaptation au drift reste concentré sur Electricity, ou s'il généralise maintenant aux nouveaux streams réels (CreditCard, HTTP).

In [ ]:
t3_lite = table3_per_stream_gain(results_df, 'Lite')
t3_adaptive = table3_per_stream_gain(results_df, 'Adaptive')
pd.concat([t3_lite, t3_adaptive], axis=1).round(4)


## 11. Table 4 — Efficacité (latence, mémoire), macro-moyennée sur les streams

In [ ]:
t4 = table4_efficiency(results_df)
t4


## 12. Graphique — Accuracy vs mémoire du pool (trade-off Lite / Adaptive)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for method, sub in t4.iterrows():
    marker = 'o' if 'DSPP' in method else ('s' if 'Static' in method else '^')
    color = 'crimson' if method in ('DSPP-Lite', 'DSPP-Adaptive') else None
    ax.scatter(sub['pool_memory_kb'], sub['accuracy'], marker=marker,
               s=80, label=method, color=color)
    ax.annotate(method, (sub['pool_memory_kb'], sub['accuracy']),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel('Mémoire du pool (KB, macro-moyenne)')
ax.set_ylabel('Accuracy (macro-moyenne)')
ax.set_title('Accuracy vs mémoire — DSPP vs ablations vs baselines')
plt.tight_layout()
plt.savefig('dspp_accuracy_vs_memory.png', dpi=150)
plt.show()


## 13. Export des tables pour le manuscrit

In [ ]:
with pd.ExcelWriter('dspp_result_tables.xlsx') as writer:
    t1_acc.round(4).to_excel(writer, sheet_name='Table1_accuracy')
    t1_kappa.round(4).to_excel(writer, sheet_name='Table1bis_kappa')
    t2.round(4).to_excel(writer, sheet_name='Table2_wilcoxon', index=False)
    pd.concat([t3_lite, t3_adaptive], axis=1).round(4).to_excel(writer, sheet_name='Table3_gain_by_stream')
    t4.round(4).to_excel(writer, sheet_name='Table4_efficiency')
    pd.DataFrame([friedman_lite]).to_excel(writer, sheet_name='Friedman', index=False)

print('Tables exportées : dspp_result_tables.xlsx')
print('Résultats bruts : dspp_raw_results.csv')
print('Figure : dspp_accuracy_vs_memory.png')


## Notes / hypothèses à vérifier avant de réutiliser ces chiffres dans le papier

- **SEA-abrupt** : implémenté via `ConceptDriftStream(width=50)` plutôt que `width=1` — `width=1` déclenche un bug de dépassement flottant côté `river` (`math.exp` overflow). `width=50` reste une transition très nette relative à `position=2500` sur un stream de 5 000 instances ; à ajuster si tu veux quelque chose d'encore plus abrupt.
- **SEA-recurring** : reconstruit "à la main" par alternance de 6 segments SEA variant 0/1 de 800 instances chacun (pas de générateur natif `river` pour du recurring multi-bascules) — à comparer avec ta version originale si tu en avais une différente.
- **DSPP-Lite** : le bagging en ligne Poisson(1) par modèle (diversity injection, Section 3.6 du papier) est implémenté avec un générateur aléatoire *privé par modèle*, seedé par `100*seed + model_id`, conformément à la note de reproductibilité de la Section 6.5.
- **Mémoire** : mesurée par taille du pickle de chaque modèle (`pickle.dumps`), en KB — c'est une proxy standard mais pas identique à un profilage mémoire runtime réel ; à noter dans la section méthodo si tu gardes cette mesure.
- **CreditCard / HTTP** : fortement déséquilibrés — la Table 1bis (Kappa) est plus informative que l'accuracy brute pour ces deux streams. Pense à les mentionner explicitement dans la partie résultats.
- Ce notebook n'a pas encore tourné en pleine échelle (10 seeds × 8 streams) dans cet environnement — seul un smoke test (2 seeds, streams courts) a été validé avant livraison. Lance `QUICK_TEST=True` d'abord pour confirmer que tout passe dans ton Colab, avant de lancer le run complet.
